# First code for analysing drop falling in marchantia cups

## 1. Documentation and Imports


Created on Mon Jan 18 2021

Author: Valentin Laplaud

The purpose of this code is to test a first way of analyzing video of water droplet
splashing in a marchantia cups. It is based on the first test experiment performed 
the 14 jan 2021

It will use image analysis done with 'Analyze Particles' in imageJ, with one text file 
per drop to analyze

In [ ]:
import matplotlib.pyplot as plt  # Plotting functions
import numpy as np # Handling numbers
import statistics as st # mean median and stuff
import scipy.signal as scpsig # Signal analysis 

## 2. Define analysis functions

### 2.1 Impact and ejection analysis

Loads the text file obtained from ImageJ for drop impact and ejection analysis

Computes vertical drop speed for impact analysis and lateral droplet speed for ejection analysis

Makes graph, saves them 

P : Path to data

CupsList : list of filenames

DropList : list of drop codes


In [ ]:
def AnalyseData(P, CupsList, DropList, FrameRate, Scale):
    

    for s in CupsList:

        ## Impact analysis
        ImpactFilename = P + '\\' + s + '-Impact_results.txt'

        F = open(ImpactFilename,'r') # Open file for reading
        lines = F.readlines() # Read all lines of the file

        # find the columns with X and Y values and Slice number
        NameVect = lines[0].split('\t') 
        nX = NameVect.index("X")
        nY = NameVect.index("Y")
        try:
            nS = NameVect.index("Slice")
        except:
            nS = NameVect.index("Slice\n") # if last column

        XposI = [] # X position of impacting drop
        YposI = [] # Y position of impacting 
        SliceI = [] # Slice list for impacting drop    

        # loops over images
        for l in lines[1:]:
            XposI.append(float(l.split('\t')[nX]))
            YposI.append(float(l.split('\t')[nY]))
            SliceI.append(float(l.split('\t')[nS]))


        YposI = np.max(YposI) - YposI   
        TimeI = np.divide(SliceI,FrameRate) # in milliseconds

        VertSpeedI = np.divide(np.diff(YposI),np.diff(TimeI))*Scale
        # vertical impact speed in mm/s

        MeanVertSpeedI = st.mean(VertSpeedI)

        # Plotting position in time with mean speed

        plt.figure()
        plt.title('Impact drop vertical position in time (mean speed = ' + str(abs(round(MeanVertSpeedI,1))) + ' px/ms)') 
        plt.xlabel('Time (ms)')
        plt.ylabel('Vertical position (px)')
        plt.plot(TimeI,YposI,'o-')
        plt.savefig(P + '\\' + s +'_FigYPos_Impact.png',dpi=300)


        ## Ejection analysis
        DropList = ['D1']
        for d in DropList:
            EjectionFilename = P + '\\' + s + '-Ejection-' + d + '_results.txt'

            F = open(EjectionFilename,'r') # Open file for reading
            lines = F.readlines() # Read all lines of the file

            # find the columns with X and Y values and Slice number
            NameVect = lines[0].split('\t') 
            nX = NameVect.index("X")
            nY = NameVect.index("Y")
            try:
                nS = NameVect.index("Slice")
            except:
                nS = NameVect.index("Slice\n") # if last column

            XposE = [] # X position of impacting drop
            YposE = [] # Y position of impacting 
            SliceE = [] # Slice list for impacting drop

            # loops over images
            for l in lines[1:]:
                XposE.append(float(l.split('\t')[nX]))
                YposE.append(float(l.split('\t')[nY]))
                SliceE.append(float(l.split('\t')[nS]))

            YposE = np.max(YposE) - YposE
            TimeE = np.divide(SliceE,FrameRate) # in milliseconds
            TimeEAxis = TimeE[0:-1]+np.diff(TimeE)/2-TimeE[0]

            VertSpeedE = np.divide(np.diff(YposE),np.diff(TimeE))*Scale
            HorzSpeedE = np.divide(np.diff(XposE),np.diff(TimeE))*Scale

            FullSpeedE = np.sqrt(np.square(VertSpeedE)+np.square(HorzSpeedE))
            SmoothFullSpeedE = scpsig.savgol_filter(FullSpeedE, 13, 3)

            MeanVertSpeedE = st.mean(VertSpeedE)
            MeanHorzSpeedE = st.mean(HorzSpeedE)

            # Plotting position in time with mean speed

            plt.figure()
            plt.title('Ejected drop ' +d + ' vertical position in time (mean speed = ' + str(abs(round(MeanVertSpeedE,1))) + ' px/ms)') 
            plt.xlabel('Time (ms)')
            plt.ylabel('Vertical position (px)')
            plt.plot(TimeE,YposE,'o-')
            plt.savefig(P + '\\' + s + '-' + d +'_FigYPos_Ejection.png',dpi=300) 

            plt.figure()
            plt.title('Ejected drop ' +d + ' Horizontal position in time (mean speed = ' + str(abs(round(MeanHorzSpeedE,1))) + ' px/ms)') 
            plt.xlabel('Time (ms)')
            plt.ylabel('Horizontal position (px)')
            plt.plot(TimeE,XposE,'o-')
            plt.savefig(P + '\\' + s + '-' + d +'_FigXPos_Ejection.png',dpi=300) 

            # Plotting full speed in time

            plt.figure()
            plt.title('Ejected drop ' +d + ' speed in time')
            plt.xlabel('Time (ms)')
            plt.ylabel('Speed (px/ms)')
            plt.plot(TimeEAxis,FullSpeedE,'o-')
            plt.plot(TimeEAxis,SmoothFullSpeedE,'r-')
            plt.savefig(P + '\\' + s + '-' + d +'_FigSpeed_Ejection.png',dpi=300) 

## X. Enter data and run

In [ ]:

# Path to data
P = r'd:\Users\laplaud\Desktop\PostDoc\Data\SplashCups\Manips\21.01.14_PremiereGouttes\Analysis'

FrameRate = 10 # in frame per millisecond 
Scale = 1 # in mn/px (here no scale where done)

CupsList = ['C1_75cm_1'] # list of cups analyzed

DropList = ['D1'] # not functional here, only a list for the first cup
# should probably be a dictionnary with for each cup with its name, and a drop list


AnalyseData(P, CupsList, DropList, FrameRate, Scale)

# Test Zone

In [ ]:
# this code works fine ! :D